In [1]:
# ===== PHASE 1: SETUP =====
!pip install ultralytics ensemble-boxes -q

import os
import pydicom
import numpy as np
import cv2
import pandas as pd
import pickle
import yaml
import torch
from pathlib import Path
from ensemble_boxes import weighted_boxes_fusion
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# GPU check
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Dataset paths
BASE_PATH = "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection"
OUTPUT_DIR = Path("/kaggle/working/dataset")

# Create folder structure
for sub in ["images/train", "images/val", "images/test",
            "labels/train", "labels/val", "labels/test"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# Load CSV
train_df = pd.read_csv(f"{BASE_PATH}/train.csv")

print(f"\n Setup complete")
print(f"   Train rows: {len(train_df)}")
print(f"   Unique images: {train_df['image_id'].nunique()}")
print(f"   Output dir: {OUTPUT_DIR}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 5.1 MB/s eta 0:00:00
GPU: Tesla T4
Memory: 15.6 GB

 Setup complete
   Train rows: 67914
   Unique images: 15000
   Output dir: /kaggle/working/dataset


In [2]:
# ===== PHASE 2: FULL DATA PIPELINE =====

# --- 2A: Image dimensions cache ---
CACHE_PATH = "/kaggle/working/image_dims.pkl"

if Path(CACHE_PATH).exists():
    with open(CACHE_PATH, 'rb') as f:
        image_dims = pickle.load(f)
    print(f"Cache loaded: {len(image_dims)} images")
else:
    print("Caching image dimensions...")
    image_dims = {}
    for img_id in tqdm(train_df['image_id'].unique(), desc="Reading DICOMs"):
        try:
            ds = pydicom.dcmread(f"{BASE_PATH}/train/{img_id}.dicom")
            h, w = ds.pixel_array.shape
            image_dims[img_id] = (w, h)
        except:
            image_dims[img_id] = (None, None)
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(image_dims, f)
    print(f"Cache saved: {len(image_dims)} images")

# --- 2B: WBF fusion ---
print("\nApplying WBF fusion...")
fused_records = []
for img_id in tqdm(train_df['image_id'].unique(), desc="Fusing"):
    if img_id not in image_dims or image_dims[img_id][0] is None:
        continue
    w, h = image_dims[img_id]
    img_df = train_df[train_df['image_id'] == img_id]
    img_df = img_df[img_df['class_name'] != 'No finding']
    if img_df.empty:
        continue
    boxes, scores, labels = [], [], []
    for _, row in img_df.iterrows():
        x1 = max(0, min(1, row['x_min'] / w))
        y1 = max(0, min(1, row['y_min'] / h))
        x2 = max(0, min(1, row['x_max'] / w))
        y2 = max(0, min(1, row['y_max'] / h))
        boxes.append([x1, y1, x2, y2])
        scores.append(1.0)
        labels.append(int(row['class_id']))
    fb, fs, fl = weighted_boxes_fusion([boxes], [scores], [labels],
                                        iou_thr=0.4, skip_box_thr=0.0)
    for b, s, l in zip(fb, fs, fl):
        fused_records.append({'image_id': img_id,
            'x_min': int(b[0]*w), 'y_min': int(b[1]*h),
            'x_max': int(b[2]*w), 'y_max': int(b[3]*h),
            'class_id': int(l)})
fused_df = pd.DataFrame(fused_records)
print(f"Fused: {len(fused_df)} boxes")

# --- 2C: Class mapping + filter ---
CLASS_ID_TO_NAME = {0:'Aortic enlargement',1:'Atelectasis',2:'Calcification',
    3:'Cardiomegaly',4:'Consolidation',5:'ILD',6:'Infiltration',7:'Lung Opacity',
    8:'Nodule/Mass',9:'Other lesion',10:'Pleural effusion',11:'Pleural thickening',
    12:'Pneumothorax',13:'Pulmonary fibrosis',14:'No finding'}
fused_df['class_name'] = fused_df['class_id'].map(CLASS_ID_TO_NAME)

SELECTED = {'Aortic enlargement':0,'Cardiomegaly':1,'Pleural thickening':2,
    'Pulmonary fibrosis':3,'Lung Opacity':4,'Other lesion':5,
    'Pleural effusion':6,'Nodule/Mass':7}
filtered_df = fused_df[fused_df['class_name'].isin(SELECTED.keys())].copy()
filtered_df['new_class_id'] = filtered_df['class_name'].map(SELECTED)
print(f"Filtered: {len(filtered_df)} boxes, {filtered_df['image_id'].nunique()} images")

# --- 2D: Train/val/test split ---
unique_images = filtered_df['image_id'].unique()
train_ids, temp_ids = train_test_split(unique_images, test_size=0.3, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)
print(f"Split: train={len(train_ids)}, val={len(val_ids)}, test={len(test_ids)}")

# Save splits + annotations
with open("/kaggle/working/splits.json", 'w') as f:
    import json
    json.dump({'train': list(train_ids), 'val': list(val_ids), 'test': list(test_ids)}, f)
filtered_df.to_csv("/kaggle/working/filtered_annotations.csv", index=False)
fused_df.to_csv("/kaggle/working/fused_annotations.csv", index=False)

# --- 2E: DICOM to PNG ---
print("\nConverting DICOM → PNG (this takes ~40 min)...")
def dicom_to_png(dcm_path, out_path):
    ds = pydicom.dcmread(str(dcm_path))
    img = ds.pixel_array.astype(float)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255.0
    img = img.astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)
    cv2.imwrite(str(out_path), img)

for split_name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    for img_id in tqdm(ids, desc=f"PNG {split_name}"):
        dcm = Path(BASE_PATH) / "train" / f"{img_id}.dicom"
        out = OUTPUT_DIR / "images" / split_name / f"{img_id}.png"
        if not out.exists():
            try:
                dicom_to_png(dcm, out)
            except Exception as e:
                pass

# --- 2F: YOLO labels ---
print("\nCreating YOLO labels...")
for split_name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    for img_id in ids:
        img_df = filtered_df[filtered_df['image_id'] == img_id]
        w, h = image_dims[img_id]
        lines = []
        for _, row in img_df.iterrows():
            cx = ((row['x_min']+row['x_max'])/2)/w
            cy = ((row['y_min']+row['y_max'])/2)/h
            bw = (row['x_max']-row['x_min'])/w
            bh = (row['y_max']-row['y_min'])/h
            if bw > 0.001 and bh > 0.001:
                lines.append(f"{int(row['new_class_id'])} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        with open(OUTPUT_DIR / "labels" / split_name / f"{img_id}.txt", 'w') as f:
            f.write('\n'.join(lines))

# --- 2G: dataset.yaml ---
cfg = {'path': str(OUTPUT_DIR), 'train': 'images/train', 'val': 'images/val',
    'test': 'images/test', 'nc': 8, 'names': list(SELECTED.keys())}
with open('/kaggle/working/dataset.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

# --- FINAL VERIFICATION ---
print("DATA PIPELINE COMPLETE")
for split in ["train", "val", "test"]:
    n_img = len(list((OUTPUT_DIR / "images" / split).glob("*.png")))
    n_lbl = len(list((OUTPUT_DIR / "labels" / split).glob("*.txt")))
    print(f"  {split}: {n_img} images, {n_lbl} labels")

Caching image dimensions...


Reading DICOMs: 100%|██████████| 15000/15000 [3:53:16<00:00,  1.07it/s]


Cache saved: 15000 images

Applying WBF fusion...


Fusing: 100%|██████████| 15000/15000 [01:26<00:00, 172.59it/s]


Fused: 22719 boxes
Filtered: 19607 boxes, 4380 images
Split: train=3066, val=657, test=657

Converting DICOM → PNG (this takes ~40 min)...


PNG test: 100%|██████████| 657/657 [12:04<00:00,  1.10s/it]



Creating YOLO labels...
DATA PIPELINE COMPLETE
  train: 3066 images, 3066 labels
  val: 657 images, 657 labels
  test: 657 images, 657 labels
